# L'importance des modèles

## Données

Repartons des données que nous avons mis en forme, et amenons progressivement une analyse plus avancée

In [6]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/pyshs/URFIST-Lyon-2026/refs/heads/main/data/css_openalex_26022026.csv")
df["title"] = df["title"].fillna("")
df["abstract"] = df["abstract"].fillna("")
df["texte"] = df["title"]+ df["abstract"]
df = df[(df["texte"].str.len() > 100) & (df["texte"].str.len() < 5000)]
df.head(3)

,id,type,primary_location,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score,abstract,journal,texte
0,https://openalex.org/W2159397589,article,"{'id': 'doi:10.1126/science.1167742', 'is_oa':...",Computational Social Science,"{'A': [0], 'field': [1], 'is': [2], 'emerging'...",2009.0,2009-02-06,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",1360.35770,A field is emerging that leverages the capacit...,Science,Computational Social ScienceA field is emergin...
2,https://openalex.org/W3081158114,article,"{'id': 'doi:10.1126/science.aaz8170', 'is_oa':...",Computational social science: Obstacles and op...,"{'Data': [0], 'sharing,': [1], 'research': [2]...",2020.0,2020-08-28,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",438.53986,"Data sharing, research ethics, and incentives ...",Science,Computational social science: Obstacles and op...
3,https://openalex.org/W3022499311,article,{'id': 'doi:10.1146/annurev-soc-121919-054621'...,Computational Social Science and Sociology,"{'The': [0], 'integration': [1], 'of': [2, 16,...",2020.0,2020-04-28,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",413.03424,The integration of social science with compute...,Annual Review of Sociology,Computational Social Science and SociologyThe ...


## Spacy pour aller vers les modèles

Installer Spacy et un modèle adapté

In [5]:
#!pip install -U spacy

In [2]:
#!pip install -U spacy
!python -m spacy download en_core_web_md

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 29.0 MB/s  0:00:01 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [21]:
import spacy

pipeline = spacy.load("en_core_web_trf")

texte = "The language Python rocks, but the day is long, It's a windy day."
doc = pipeline(texte)

In [22]:
type(doc)

spacy.tokens.doc.Doc

In [23]:
list(doc)

[The,
 language,
 Python,
 rocks,
 ,,
 but,
 the,
 day,
 is,
 long,
 ,,
 It,
 's,
 a,
 windy,
 day,
 .]

Le texte est "structuré" : composé de différents éléments qui n'existaient pas avant

In [ ]:
doc[0]

POS, lemmatisation, etc.

In [24]:
for token in doc:
    print(token.text, token.lemma_, token.pos_, token.dep_, token.lemma_)

The the DET det the
language language NOUN nsubj language
Python Python PROPN appos Python
rocks rock VERB ROOT rock
, , PUNCT punct ,
but but CCONJ cc but
the the DET det the
day day NOUN nsubj day
is be AUX ccomp be
long long ADJ acomp long
, , PUNCT punct ,
It it PRON nsubj it
's be AUX conj be
a a DET det a
windy windy ADJ amod windy
day day NOUN attr day
. . PUNCT punct .


Récupérer uniquement les verbes ?

In [28]:
liste = []
for element in doc:
    if element.pos_ in ["VERB","AUX"]:
        liste.append(element.text)
liste

['rocks', 'is', "'s"]

List comprehension

In [30]:
[e.text for e in doc if e.pos_ in ["VERB","AUX"]]

['rocks', 'is', "'s"]

NER

In [34]:
for ent in doc.ents:
    print(ent.text, ent.label_)

Python LANGUAGE
a windy day DATE


In [37]:
for i in pipeline("We the March 27th, and we are doing some Python").ents:
    print(i.text, i.label_)

the March 27th DATE
Python WORK_OF_ART


In [41]:
result = df["texte"][:10].apply(pipeline)

In [43]:
for r in result:
    print(r.ents)

()
()
()
(zero, zero, zero, 13, 25, English, today, two, zero, 2)
(Galileo, 2010, John Wiley &amp, Sons, Inc.)
(first, second)
(Computational Social Science,)
(the last half century,)
(three,)
(two, Google, Facebook)


In [47]:
from spacy import displacy
displacy.render(result.iloc[3], style="ent", 
                jupyter=True)

Plongements

In [53]:
# Compare two documents
doc1 = pipeline(df["abstract"].iloc[0])
doc2 = pipeline(df["abstract"].iloc[3])
print(doc1.similarity(doc2))

0.0


/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_47649/3396008260.py:4: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  print(doc1.similarity(doc2))
/var/folders/f9/d2d_05ws5gncmml0fx0c00kw0000gp/T/ipykernel_47649/3396008260.py:4: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  print(doc1.similarity(doc2))


## Utiliser des modèles d'HuggingFace : le cas de GliNER

Aller vers les modèles :
- HuggingFace
- Cartes des modèles
- Transformers

Un modèle plus complexe pour l'identification d'entités

Par exemple : https://huggingface.co/urchade/gliner_multi_pii-v1

In [1]:
#!pip install gliner
from gliner import GLiNER

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [4]:
text = "Il se trouve qu'Émilien aime faire du Python"

labels = ["Personne nommée", "Langage de programmation"]

entities = model.predict_entities(text, labels)

entities

[{'start': 16,
  'end': 23,
  'text': 'Émilien',
  'label': 'Personne nommée',
  'score': 0.8683179616928101},
 {'start': 38,
  'end': 44,
  'text': 'Python',
  'label': 'Langage de programmation',
  'score': 0.990264892578125}]

In [8]:
r = df["texte"][0:10].apply(lambda x:  model.predict_entities(x, ["digital method"]))
r

0                                                    []
2                                                    []
3     [{'start': 204, 'end': 225, 'text': 'computati...
5                                                    []
7     [{'start': 415, 'end': 435, 'text': 'advanced ...
9     [{'start': 3, 'end': 23, 'text': 'agent-based ...
11                                                   []
13    [{'start': 404, 'end': 427, 'text': 'agent-bas...
15    [{'start': 91, 'end': 111, 'text': 'Agent Base...
20    [{'start': 137, 'end': 165, 'text': 'computati...
Name: texte, dtype: object

In [11]:
#list(r)

## Application

Faire de l'analyse de sentiments

Une question : **quelles sont les prises de paroles les plus négatives ?**

- Embarras du choix
    - Par ex : [🚀 distilbert-based Multilingual Sentiment Classification Model
](https://huggingface.co/tabularisai/multilingual-sentiment-analysis)
- Comprendre le modèle / ce qu'il fait
- Importance d'évaluer son résultat